# Gerador de fatores de correção IPCA

Este notebook transforma **números-índice do IPCA** (retirados do IBGE) em uma tabela de
**fatores de correção monetária**, gravada em `ipca_ibge_graf_cgu.csv` para uso no seu
pipeline de deflação das remunerações.

**O que você faz:** cola os números-índice na célula de entrada e escolhe o ano-base.
**O que sai:** um CSV com as colunas `ano` e `fator_correcao`.

---

### De onde tirar os índices

Fonte: **IBGE — SIDRA, Tabela 1737** (IPCA, série histórica com número-índice,
base dezembro/1993 = 100).

1. Acesse `https://sidra.ibge.gov.br/tabela/1737`
2. Em **Variável**, selecione *"IPCA - Número-índice (base: dezembro de 1993 = 100)"*.
3. Em **Mês**, marque **novembro** de cada ano que você usa (mês de referência das
   tabelas `{ano}_nov_...`).
4. Copie os valores para a lista `indices` abaixo.

> **Fórmula:** `fator(ano) = índice(ano_base) / índice(ano)`
> Fator `> 1` para anos anteriores à base; `= 1` no ano-base; `< 1` para anos posteriores.


## 1. Entrada — cole aqui os números-índice do IBGE

In [ ]:
# Números-índice do IPCA (mês de novembro), base dez/1993 = 100.
# Formato: (ano, numero_indice). Aceita ponto OU vírgula decimal.
#
# >>> SUBSTITUA pelos valores oficiais copiados da Tabela 1737 do SIDRA <<<
indices = [
    (2020, "5486.52"),
    (2021, "6075.69"),
    (2022, "6434.20"),
    (2023, "6735.55"),
    (2024, "7063.77"),
    (2025, "7378.94"),
]

# Ano que servirá de base (fator = 1,0 nesse ano).
# Deixe como o ano mais recente para expressar tudo em "reais de hoje".
ano_base = 2025

# Nome do arquivo de saída (mesmo nome esperado pelo seu pipeline).
arquivo_saida = "ipca_ibge_graf_cgu.csv"

# Casas decimais dos fatores no CSV.
casas_decimais = 4

## 2. Processamento

In [ ]:
import pandas as pd

def _num(x):
    """Converte '6.944,83' ou '6944.83' ou float em float."""
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip()
    # se tem virgula, tratamos virgula como decimal e ponto como milhar
    if "," in s:
        s = s.replace(".", "").replace(",", ".")
    return float(s)

# monta dicionario ano -> indice (float)
idx = {int(ano): _num(val) for ano, val in indices}

# validacoes basicas
assert idx, "A lista 'indices' esta vazia."
assert ano_base in idx, f"O ano_base ({ano_base}) nao esta na lista de indices."
assert len(idx) == len({a for a, _ in indices}), "Ha anos duplicados em 'indices'."

I_base = idx[ano_base]

# calcula os fatores
registros = []
for ano in sorted(idx):
    fator = I_base / idx[ano]
    registros.append({"ano": ano, "fator_correcao": round(fator, casas_decimais)})

df_fatores = pd.DataFrame(registros)
df_fatores

,ano,fator_correcao
0,2020,1.3449
1,2021,1.2145
2,2022,1.1468
3,2023,1.0955
4,2024,1.0446
5,2025,1.0000


## 3. Saída — grava o CSV

In [ ]:
df_fatores.to_csv(arquivo_saida, index=False)
print(f"Arquivo gravado: {arquivo_saida}")
print(f"{len(df_fatores)} anos ({anos_ord[0]}–{anos_ord[-1]}), base {ano_base} = 1,0")
df_fatores

Arquivo gravado: ipca_ibge_graf_cgu.csv
6 anos (2020–2025), base 2025 = 1,0


,ano,fator_correcao
0,2020,1.3449
1,2021,1.2145
2,2022,1.1468
3,2023,1.0955
4,2024,1.0446
5,2025,1.0000


## 6. (Opcional) Rebasear para outro ano sem recopiar índices

Se mais tarde você quiser mudar a base (por exemplo, de 2025 para 2026) sem mexer na lista,
basta alterar `ano_base` na célula 1 e reexecutar — os índices são a fonte da verdade e o
fator é recalculado automaticamente. Não é necessário nenhum passo extra de rebaseamento.

---

### Como usar no seu pipeline

O arquivo gerado tem exatamente as colunas que seu código consome:

```python
ipca = pd.read_csv('ipca_ibge_graf_cgu.csv')
df1 = df.merge(ipca[['ano', 'fator_correcao']], how='left', on='ano')
df1['media_corrigido']   = df1['media']   * df1['fator_correcao']
df1['mediana_corrigido'] = df1['mediana'] * df1['fator_correcao']
```

Basta manter `ipca_ibge_graf_cgu.csv` na mesma pasta do notebook de análise.
